# LangChain и Deep Agents

**Курс «Разработка и внедрение AI-агентов», МФТИ.**

На прошлом занятии мы ответили на вопрос

> **Поместятся ли 28 студентов в аудитории A-101?**

вручную. Приложение отправляло модели вопрос и описание функции, получало вызов `get_room_info`,
проверяло его, исполняло функцию, возвращало результат и делало второй запрос. Для вопроса про две
аудитории те же шаги пришлось повторять в цикле. Последний слайд занятия задавал вопрос: кто будет
вести этот цикл, когда задач станет больше?

Сегодня цикл берёт на себя фреймворк. Сначала — **LangChain**: функция `create_agent` строит агента,
который сам повторяет шаги «модель → инструмент → модель». Затем разберём, как вмешиваться в работу
агента через **middleware**, и посмотрим на **Deep Agents** — готовую обвязку для длинных задач:
с планом, файлами, командной строкой, субагентами и памятью.

### Маршрут занятия

1. **Подготовка.** Установка и настройки.
2. **`create_agent`.** Тот же обмен с функцией одной строкой.
3. **Что внутри агента.** Граф LangGraph, поток шагов, память диалога.
4. **Middleware.** Хуки агентного цикла и свои правила.
5. **Встроенные middleware.** Лимиты и подтверждение действий человеком.
6. **Особенности GigaChat.** Что модель и адаптер не поддерживают и как это обойти.
7. **Deep Agents.** Что такое обвязка и какой агент получается одной функцией.
8. **План.** Список задач `write_todos`.
9. **Командная строка.** Агент пишет и запускает скрипты.
10. **Субагенты.** Делегирование части работы.
11. **Память.** Файл `AGENTS.md` с правилами проекта.
12. **Практика.**
13. **Итоги.**

### Что нужно знать заранее

Материал прошлого занятия: сообщения `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`,
вызов функции и её результат, `bind_tools`. Ячейки выполняются по порядку. Запросы идут к настоящему
API GigaChat и расходуют токены. Агенты в разделах 7–12 делают по нескольку запросов за раз.

<details><summary>Словарь терминов</summary>

- **Агент** — программа, в которой модель сама решает, какие инструменты вызвать и когда закончить.
- **Агентный цикл (agent loop)** — повторение шагов «модель → инструменты → модель», пока модель
  не ответит без вызовов.
- **Граф** — описание программы как набора узлов (шагов) и переходов между ними. LangGraph исполняет
  такие графы и хранит их состояние.
- **Состояние (state)** — данные, которые узлы графа читают и изменяют; у агента это прежде всего
  список сообщений.
- **Middleware** — код, который встраивается в агентный цикл в определённых точках и может прочитать
  или изменить запрос, ответ, вызов инструмента.
- **Хук** — точка, в которой фреймворк вызывает ваш код.
- **Обвязка (harness)** — всё, что окружает модель в агенте: инструменты, среда исполнения, правила,
  управление контекстом.
- **Субагент** — отдельный агент, которому главный агент поручает часть работы.

</details>

## 1. Подготовка

### 1.1. Установка

Окружение ставится так же, как на прошлом занятии, — через [uv](https://docs.astral.sh/uv/getting-started/installation/).
Команды одинаковы для macOS, Linux и Windows:

```bash
cd mipt_llm_api
git pull
cd langchain_deep_agents
uv sync
cp .env.example .env          # Windows: copy .env.example .env
uv run jupyter lab langchain_deep_agents.ipynb
```

Если репозитория ещё нет, сначала `git clone https://github.com/trashchenkov/mipt_llm_api`.

Прокси GPT2GIGA сегодня не нужен: все примеры работают с GigaChat через `langchain-gigachat`.

| Пакет | Версия | Зачем |
|---|---|---|
| `langchain` | 1.4.3 | `create_agent` и middleware |
| `langgraph` | 1.2.12 | граф, состояние, память диалога |
| `langchain-gigachat` | 0.5.1 | модель GigaChat в объектах LangChain — тот же адаптер, что на прошлом занятии |
| `deepagents` | 0.7.21 | обвязка Deep Agents |
| `deepagents-gigachat` | 0.0.4 | профиль обвязки под GigaChat |

### 1.2. Настройки

В `.env` нужен тот же ключ авторизации, что на прошлом занятии. Его можно скопировать из файла
`.env` папки `llm_api_to_langchain`. Проверить окружение:

```bash
uv run tools/check_env.py
```

Ячейка ниже загружает `.env` и создаёт модель. `GigaChat()` без аргументов читает ключ, scope и
название модели из переменных `GIGACHAT_*`.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_gigachat import GigaChat

ROOT = Path.cwd()
assert (ROOT / "room_tool.py").is_file(), "Запустите Jupyter из папки langchain_deep_agents"
load_dotenv(ROOT / ".env", override=False)
assert os.getenv("GIGACHAT_CREDENTIALS"), "Заполните GIGACHAT_CREDENTIALS в .env"

llm = GigaChat(timeout=120)
print("модель:", llm.model or os.getenv("GIGACHAT_MODEL"))

### 1.3. Справочник и вспомогательная функция

Справочник аудиторий тот же, что на прошлом занятии: `room_tool.py` с аудиториями A-101 и B-202.
Функцию `get_room_info` превращаем в инструмент LangChain через `tool(...)`: описание и схема
аргументов строятся по сигнатуре и докстрингу.

Агент возвращает длинный список сообщений. Функция `show_steps` печатает его коротко: тип сообщения,
вызовы инструментов и начало текста. Запросов к API она не делает.

In [ ]:
from langchain_core.tools import tool

from room_tool import get_room_info

room_tool = tool(get_room_info)
print(room_tool.name, "—", room_tool.description)
print("аргументы:", room_tool.args)


def show_steps(messages):
    for message in messages:
        kind = type(message).__name__
        calls = getattr(message, "tool_calls", None) or []
        if calls:
            for call in calls:
                args = str(call["args"])
                print(f"{kind:13} → {call['name']}({args[:100] + '…' if len(args) > 100 else args})")
        else:
            print(f"{kind:13}   {str(message.content)[:110]!r}")

## 2. `create_agent`: цикл, который раньше писали руками

### 2.1. Агент одной функцией

На прошлом занятии цикл «вызов → исполнение → результат → снова модель» занимал полтора десятка
строк. В LangChain его строит функция [`create_agent`](https://docs.langchain.com/oss/python/langchain/agents):
она принимает модель, список инструментов и системный промпт и возвращает готового агента.

Агент вызывается методом `invoke`. На вход — словарь состояния с ключом `messages`, на выходе —
состояние после работы агента, в котором `messages` содержит всю историю: вопрос, вызовы, результаты
и ответ.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[room_tool], system_prompt="Отвечай кратко по-русски.")
result = agent.invoke({"messages": [{"role": "user", "content": "Сравни вместимость A-101 и B-202."}]})
show_steps(result["messages"])

Сравните с разделом 7.7 прошлого занятия. Шаги те же:

1. модель просит `get_room_info` для A-101;
2. агент исполняет функцию и добавляет `ToolMessage` с результатом;
3. модель просит B-202, агент снова исполняет;
4. модель отвечает текстом — вызовов больше нет, цикл заканчивается.

Изменилось только то, кто ведёт цикл. Проверять аргументы, исполнять функцию, складывать историю и
решать, когда остановиться, теперь будет агент. Системный промпт в истории не виден: агент добавляет
его к каждому запросу к модели сам и в состояние не сохраняет.

### 2.2. Что делает агент с ошибкой инструмента

Спросим про аудиторию, которой нет в справочнике. `get_room_info` в этом случае выбрасывает
`ValueError`. Ячейка перехватывает исключение, чтобы показать, что произошло.

In [ ]:
try:
    agent.invoke({"messages": [{"role": "user", "content": "Сколько мест в C-303?"}]})
except ValueError as error:
    print("запуск агента прерван:", error)

По умолчанию исключение внутри инструмента прерывает весь запуск агента: модель не узнаёт об
ошибке и не может ответить пользователю. На прошлом занятии проверку делал наш код вокруг вызова.
В агенте для этого есть middleware — к этой задаче вернёмся в практике (раздел 12).

## 3. Что внутри агента

### 3.1. Граф

`create_agent` возвращает объект `CompiledStateGraph` из [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview).
Агент — это граф из двух рабочих узлов:

- `model` — вызывает модель с историей сообщений;
- `tools` — исполняет вызовы инструментов из последнего ответа модели.

После `model` граф смотрит на ответ: есть вызовы — переход в `tools`, нет — конец. После `tools` —
снова `model`. Это и есть агентный цикл, записанный в виде графа.

In [ ]:
print(type(agent).__name__)
print("узлы:", list(agent.get_graph().nodes))
print(agent.get_graph().draw_mermaid())

Текст после узлов — описание графа на языке [Mermaid](https://mermaid.js.org/). Его можно вставить в
[mermaid.live](https://mermaid.live), чтобы увидеть картинку: пунктирные стрелки — условные переходы.

### 3.2. Поток шагов

`invoke` возвращает результат, когда агент закончил. Метод `stream` отдаёт работу по шагам: после
каждого узла — что этот узел добавил в состояние. Так видно, какой узел что сделал. Чат-интерфейсы
используют этот режим, чтобы показывать пользователю ход работы агента.

In [ ]:
question = {"messages": [{"role": "user", "content": "Сколько мест в A-101?"}]}
for update in agent.stream(question, stream_mode="updates"):
    for node, state in update.items():
        print(f"узел {node}:")
        show_steps(state["messages"])

### 3.3. Память диалога

Агент из раздела 2 ничего не помнит между вызовами `invoke`: каждый вызов начинается с пустой
истории. Чтобы агент продолжал разговор, нужны две вещи:

- **checkpointer** — хранилище состояния графа. `InMemorySaver` держит его в памяти процесса;
  для рабочих систем есть хранилища в базах данных;
- **`thread_id`** — идентификатор диалога. Состояние сохраняется отдельно для каждого `thread_id`.

Вызов с тем же `thread_id` продолжает историю, с новым — начинает заново.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

chat_agent = create_agent(model=llm, tools=[room_tool], checkpointer=InMemorySaver(),
                          system_prompt="Отвечай кратко по-русски.")
thread_1 = {"configurable": {"thread_id": "rooms-1"}}
chat_agent.invoke({"messages": [{"role": "user", "content": "Сколько мест в A-101?"}]}, thread_1)
same = chat_agent.invoke({"messages": [{"role": "user", "content": "А проектор там есть?"}]}, thread_1)
print("тот же thread_id:", same["messages"][-1].content)
print("сообщений в истории:", len(same["messages"]))

other = chat_agent.invoke({"messages": [{"role": "user", "content": "А проектор там есть?"}]},
                          {"configurable": {"thread_id": "rooms-2"}})
print("новый thread_id:", other["messages"][-1].content)

В том же диалоге агент понимает, что «там» — это A-101: в историю попали и прошлый вопрос, и
результат функции. В новом диалоге истории нет, и модель просит уточнить, о какой аудитории речь.

Память диалога — это история сообщений, которую агент передаёт модели целиком в каждом запросе. Чем
длиннее диалог, тем больше входных токенов в каждом следующем запросе. Как с этим справляются
Deep Agents, увидим в разделе 7.

## 4. Middleware: правила вокруг агентного цикла

### 4.1. Зачем вмешиваться в цикл

Агент из раздела 2 делает всё сам. На практике нужно управлять его работой: ограничить число шагов,
не дать выполнить опасное действие без подтверждения, сократить разросшуюся историю, обработать
ошибку инструмента, записать, что происходило. В LangChain 1.x всё это делается одним механизмом —
[middleware](https://docs.langchain.com/oss/python/langchain/middleware/overview).

Middleware — объект с методами-хуками. Агент вызывает их в определённых точках цикла:

| Хук | Когда вызывается | Типичное применение |
|---|---|---|
| `before_agent` | один раз, в начале запуска | проверка входа, режимы работы |
| `before_model` | перед каждым запросом к модели | сокращение истории, добавление контекста |
| `wrap_model_call` | вокруг каждого запроса к модели | повтор, замена модели, изменение списка инструментов |
| `after_model` | после каждого ответа модели | проверка ответа, метрики шага |
| `wrap_tool_call` | вокруг каждого вызова инструмента | ошибки, таймауты, проверка аргументов |
| `after_agent` | один раз, в конце запуска | аудит, сохранение результатов |

Хуки `before_*` и `after_*` получают состояние агента и могут вернуть изменения. Хуки `wrap_*` получают
запрос и функцию `handler`, которая выполняет сам вызов: middleware решает, вызвать ли `handler`, с
какими данными и что сделать с результатом.

### 4.2. Журнал хуков

Проще всего увидеть порядок, записав каждый вызов хука. Middleware можно объявить классом-наследником
`AgentMiddleware` или короче — декораторами `@before_model`, `@wrap_tool_call` и другими. Используем
декораторы.

In [ ]:
import time

from langchain.agents.middleware import (
    after_model,
    before_agent,
    before_model,
    wrap_model_call,
    wrap_tool_call,
)

hook_log = []


@before_agent
def log_before_agent(state, runtime):
    hook_log.append(f"before_agent: сообщений {len(state['messages'])}")


@before_model
def log_before_model(state, runtime):
    hook_log.append(f"before_model: сообщений {len(state['messages'])}")


@wrap_model_call
def log_model_call(request, handler):
    started = time.perf_counter()
    response = handler(request)
    hook_log.append(f"wrap_model_call: {time.perf_counter() - started:.1f} с, инструментов {len(request.tools)}")
    return response


@after_model
def log_after_model(state, runtime):
    hook_log.append(f"after_model: вызовов в ответе {len(state['messages'][-1].tool_calls)}")


@wrap_tool_call
def log_tool_call(request, handler):
    hook_log.append(f"wrap_tool_call: {request.tool_call['name']}({request.tool_call['args']})")
    return handler(request)


logged_agent = create_agent(model=llm, tools=[room_tool], middleware=[
    log_before_agent, log_before_model, log_model_call, log_after_model, log_tool_call])
logged_agent.invoke({"messages": [{"role": "user", "content": "Сколько мест в A-101?"}]})
print("\n".join(hook_log))
print("\nузлы графа:", list(logged_agent.get_graph().nodes))

Что видно:

- `before_agent` сработал один раз, остальные — на каждом шаге цикла;
- модель вызывалась дважды: первый раз ответила вызовом функции, второй — текстом;
- `wrap_tool_call` сработал между ними, с именем и аргументами вызова;
- в графе появились узлы `…before_agent`, `…before_model`, `…after_model`. Хуки `before_*` и
  `after_*` становятся отдельными узлами графа, а `wrap_*` оборачивают уже существующие узлы
  `model` и `tools`.

Если middleware несколько, `before_*` выполняются в порядке списка, `after_*` — в обратном, а
`wrap_*` вкладываются друг в друга: первый в списке оказывается внешней обёрткой.

## 5. Встроенные middleware

В LangChain есть готовые middleware для частых задач: лимиты, суммаризация истории, повтор при
ошибке, замена модели, поиск персональных данных, подтверждение человеком. Список —
в [документации](https://docs.langchain.com/oss/python/langchain/middleware/built-in).
Посмотрим на два случая.

### 5.1. Лимиты

Агент может зациклиться: снова и снова вызывать один инструмент. Лимиты обрывают такой цикл.

- `ToolCallLimitMiddleware` ограничивает число вызовов инструментов. Вызов сверх лимита не
  исполняется: агент получает вместо результата сообщение о превышении и продолжает работу.
- `ModelCallLimitMiddleware` ограничивает число запросов к модели. При `exit_behavior="end"` агент
  останавливается и последним сообщением пишет, что лимит исчерпан.

Зададим заведомо малые лимиты для вопроса про две аудитории.

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware

two_rooms = {"messages": [{"role": "user", "content": "Сравни вместимость A-101 и B-202."}]}

limited = create_agent(model=llm, tools=[room_tool], middleware=[ToolCallLimitMiddleware(run_limit=1)])
show_steps(limited.invoke(two_rooms)["messages"])
print()
limited = create_agent(model=llm, tools=[room_tool],
                       middleware=[ModelCallLimitMiddleware(run_limit=2, exit_behavior="end")])
show_steps(limited.invoke(two_rooms)["messages"])

В первом случае второй вызов получил `ToolMessage` с текстом о превышении лимита, и модель ответила
тем, что успела узнать. Во втором агент остановился после двух запросов к модели, не дав ей ответить.
Лимит — защита от зацикливания и перерасхода токенов; значения подбирают с запасом под реальные задачи.

### 5.2. Подтверждение человеком

Некоторые действия агента нельзя выполнять без согласия человека: отправка письма, оплата, удаление
файлов. `HumanInTheLoopMiddleware` приостанавливает агента перед вызовом указанного инструмента.

Пауза устроена через механизм прерываний LangGraph:

1. `invoke` возвращает состояние с ключом `__interrupt__`, где описано, какой вызов ждёт решения;
2. состояние сохраняется в checkpointer, поэтому без него прерывание не работает;
3. человек принимает решение, и агент продолжает работу вызовом `invoke(Command(resume=...))` с тем же
   `thread_id`. Решения: `approve` — выполнить, `edit` — изменить аргументы, `reject` — отклонить,
   `respond` — ответить вместо инструмента.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

careful_agent = create_agent(model=llm, tools=[room_tool], checkpointer=InMemorySaver(),
                             middleware=[HumanInTheLoopMiddleware(interrupt_on={"get_room_info": True})])
thread = {"configurable": {"thread_id": "approve-1"}}

paused = careful_agent.invoke({"messages": [{"role": "user", "content": "Сколько мест в A-101?"}]}, thread)
request = paused["__interrupt__"][0].value["action_requests"][0]
print("ждёт решения:", request["name"], request["args"])

resumed = careful_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), thread)
show_steps(resumed["messages"])

Между двумя вызовами `invoke` может пройти сколько угодно времени: агент «спит» в checkpointer.
В приложении здесь показывают пользователю кнопки «Разрешить» и «Отклонить».

## 6. Особенности GigaChat

Middleware удобны и для того, чтобы приспособить агента к конкретной модели. У GigaChat есть
несколько ограничений, о которые агенты спотыкаются.

### 6.1. Нет режима «вызови какую-нибудь функцию»

В API OpenAI есть режим `tool_choice="required"` (в LangChain — `"any"`): модель обязана вызвать
одну из функций, но сама выбирает какую. GigaChat умеет только `auto` или принудительный вызов
конкретной функции по имени. Адаптер `langchain-gigachat` отказывается от `"any"` сразу.

In [ ]:
for choice in ("any", "get_room_info"):
    try:
        reply = llm.bind_tools([room_tool], tool_choice=choice).invoke("Привет!")
        print(f"tool_choice={choice!r}: вызов {reply.tool_calls[0]['name']}({reply.tool_calls[0]['args']})")
    except ValueError as error:
        print(f"tool_choice={choice!r}: {str(error)[:110]}…")

Принудительный вызов конкретной функции на «Привет!» даёт придуманный аргумент — это мы видели и на
прошлом занятии.

Обойти ограничение можно middleware: если модель ответила текстом без вызова, повторить запрос с
напоминанием. Важная деталь: `wrap_model_call` срабатывает на **каждом** запросе к модели, в том
числе на последнем — когда результаты функций уже получены и модель должна ответить текстом. Если
требовать вызов и там, агент никогда не закончит работу. Поэтому правило действует, только пока в
истории нет результата функции.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage


@wrap_model_call
def require_tool_call(request, handler):
    response = handler(request)
    if isinstance(request.messages[-1], ToolMessage):
        return response  # результаты функций уже есть: модель вправе ответить текстом
    for attempt in range(2):
        if response.result[-1].tool_calls:
            break
        names = ", ".join(t.name for t in request.tools)
        print(f"  повтор {attempt + 1}: вызова нет")
        reminder = HumanMessage(f"Не отвечай текстом. Вызови одну из функций: {names}.")
        response = handler(request.override(messages=[*request.messages, reminder]))
    return response


strict_agent = create_agent(model=llm, tools=[room_tool], middleware=[require_tool_call])
show_steps(strict_agent.invoke({"messages": [{"role": "user", "content": "Привет! Какие аудитории ты знаешь?"}]})["messages"])

`request.override(...)` создаёт копию запроса с изменёнными полями; исходный запрос не меняется.
Напоминание добавлено к копии запроса, поэтому в историю агента оно не попадает.

Повтор не гарантирует вызов. На вопросы, где данные справочника действительно нужны, GigaChat
вызывает функцию сам. На «Какие аудитории ты знаешь?» подходящего аргумента нет, и модель при
подготовке занятия отвечала текстом даже после двух напоминаний. Middleware повышает шанс, а
окончательно решает модель.

### 6.2. Системное сообщение — только первым

GigaChat принимает `SystemMessage` только в начале истории. Системное сообщение в середине диалога —
ошибка `400`. Это важно для middleware, которые добавляют подсказки по ходу работы агента: такие
подсказки отправляют как `HumanMessage`.

In [ ]:
from langchain_core.messages import AIMessage, SystemMessage

history = [SystemMessage("Отвечай кратко."), HumanMessage("Привет"), AIMessage("Здравствуйте!"),
           SystemMessage("Отвечай одним словом."), HumanMessage("Как дела?")]
try:
    llm.invoke(history)
except Exception as error:  # noqa: BLE001 — показываем ответ сервера
    print(type(error).__name__, str(error)[:150])

### 6.3. Типы-объединения в аргументах

Аргумент вида `capacity: int | str` описывается в JSON Schema через `anyOf` — «одно из». Что с ним
происходит, проверено при подготовке занятия:

| Где | Что происходит |
|---|---|
| API v2 | принимает, модель передаёт корректное `{"capacity": 30}` |
| API v1 | принимает, но аргумент приходит обёрнутым: `{"capacity": {"value": 30}}` — не подходит ни под один тип |
| `langchain-gigachat` 0.5.1 (работает через v1) | отклоняет схему ещё до отправки |

In [ ]:
from langchain_gigachat.utils.function_calling import IncorrectSchemaException


@tool
def find_room(capacity: int | str) -> str:
    """Найти аудиторию по вместимости: числом или словами."""
    return "B-202"


print("схема:", find_room.args)
try:
    create_agent(model=llm, tools=[room_tool, find_room]).invoke(
        {"messages": [{"role": "user", "content": "Сколько мест в A-101?"}]})
except IncorrectSchemaException as error:
    print("IncorrectSchemaException:", str(error).splitlines()[-1])

Запрос падает, даже когда сама функция `find_room` для ответа не нужна: адаптер проверяет все
инструменты из списка. Если инструменты приходят извне — например, из чужой библиотеки или MCP-сервера
(о нём — на следующих занятиях), — middleware может убрать неподходящие перед каждым запросом:

In [ ]:
def has_union(schema):
    if isinstance(schema, dict):
        return "anyOf" in schema or "oneOf" in schema or any(has_union(v) for v in schema.values())
    if isinstance(schema, list):
        return any(has_union(v) for v in schema)
    return False


@wrap_model_call
def drop_union_tools(request, handler):
    kept = [t for t in request.tools if not has_union(getattr(t, "args", {}))]
    return handler(request.override(tools=kept))


safe_agent = create_agent(model=llm, tools=[room_tool, find_room], middleware=[drop_union_tools])
show_steps(safe_agent.invoke({"messages": [{"role": "user", "content": "Сколько мест в A-101?"}]})["messages"])

Другой путь — упростить схему: заменить объединение одним типом, например строкой, и разбирать
значение внутри функции. Какой путь выбрать, зависит от того, можно ли менять сам инструмент.

## 7. Deep Agents: обвязка для длинных задач

### 7.1. Чего не хватает агенту из `create_agent`

Агент из раздела 2 хорошо справляется с короткими задачами: два-три вызова, и ответ готов. На длинных
задачах — разобрать проект, собрать отчёт из десятка источников, написать и отладить скрипт — у него
появляются четыре проблемы:

1. **Контекст переполняется.** Каждый результат инструмента остаётся в истории и уходит в каждый
   следующий запрос к модели.
2. **План теряется.** Через двадцать шагов модель забывает, что собиралась сделать.
3. **Инструкции не масштабируются.** Все правила и процедуры приходится держать в системном промпте.
4. **Нет рабочей среды.** Агенту некуда сохранить промежуточный результат и негде запустить код.

Слой вокруг модели, который решает эти проблемы, называют **обвязкой** (harness). Такие обвязки стоят
внутри Claude Code, Codex CLI и других агентов для программирования. [Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) —
открытая обвязка от команды LangChain, собранная из тех же middleware, что мы видели в разделах 4–5.

| Слой | Что даёт |
|---|---|
| **Deep Agents** | готовая обвязка: файлы, командная строка, план, субагенты, память, skills |
| **LangChain** | `create_agent`, инструменты, middleware |
| **LangGraph** | граф, состояние, checkpointer, прерывания, поток шагов |
| **langchain-core** | сообщения, модели, инструменты |

### 7.2. Агент одной функцией

`create_deep_agent` принимает те же модель и инструменты, что `create_agent`. Для модели GigaChat он
сам подключает **профиль** из пакета `deepagents-gigachat` — набор промптов и middleware, настроенных
под привычки GigaChat. При подключении профиль печатает строку `Harness profile loaded`. Как
профиль собирали и что он даёт, описано в статье
[«Профиль GigaChat для Deep Agents»](https://habr.com/ru/companies/sberbank/articles/1073668/).

In [ ]:
from deepagents import create_deep_agent

deep_agent = create_deep_agent(model=llm, tools=[room_tool])
print("инструменты:", list(deep_agent.nodes["tools"].bound.tools_by_name))
print("узлы:", [n for n in deep_agent.get_graph().nodes if "." in n])

Мы передали один инструмент, а у агента их одиннадцать:

| Инструмент | Что делает |
|---|---|
| `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, `grep` | работа с файлами |
| `execute` | команда в командной строке — модель получает его, только если бэкенд её поддерживает (раздел 9) |
| `task` | поручить часть работы субагенту (раздел 10) |
| `think` | записать рассуждение без действия — его добавляет профиль GigaChat |
| `get_room_info` | наш справочник |

Узлы с точкой в имени — хуки middleware обвязки и профиля. Например, `LoopBreakerMiddleware` из
профиля распознаёт зацикливание — три одинаковых вызова подряд, три ошибки подряд — и подсказывает
модели сменить стратегию.

Системный промпт Deep Agent с профилем — около 20 тысяч символов: правила работы с файлами,
командной строкой, памятью, бюджетом шагов. Это плата за поведение в длинных задачах: даже короткий
вопрос стоит больше входных токенов, чем у агента из раздела 2.

### 7.3. Файлы

По умолчанию файлы агента живут в его **состоянии**: это бэкенд `StateBackend`. Агент пишет файл
инструментом `write_file`, файл попадает в `result["files"]`, на диск ничего не записывается. Такие
файлы — рабочая память агента внутри одного запуска: сюда он складывает промежуточные результаты,
вместо того чтобы держать их в истории сообщений.

In [ ]:
result = deep_agent.invoke({"messages": [{"role": "user", "content":
    "Проверь аудитории A-101 и B-202 для группы из 28 человек и сохрани итог в файл report.md."}]})
show_steps(result["messages"])
for name, file in result["files"].items():
    print(f"\n--- {name}\n{file['content'][:600]}")

Кроме `StateBackend` есть другие бэкенды:

| Бэкенд | Где файлы |
|---|---|
| `StateBackend` | в состоянии графа — по умолчанию |
| `FilesystemBackend` | на диске, в папке `root_dir` |
| `LocalShellBackend` | на диске, плюс инструмент `execute` для командной строки |
| `StoreBackend` | в хранилище LangGraph Store — переживают перезапуск |
| `CompositeBackend` | разные бэкенды для разных путей, например `/memories/` — в Store, остальное — в состоянии |

## 8. План: список задач

Для задач из многих шагов в Deep Agents есть инструмент `write_todos`: модель записывает план списком,
отмечает пункты как `in_progress` и `completed` и сверяется с ним по ходу работы. Список хранится в
состоянии агента, в `result["todos"]`.

В `deepagents` 0.7 этот инструмент стал необязательным: его подключают явно, передав
`TodoListMiddleware` в `middleware`. Вести план GigaChat начинает, когда об этом прямо попросить.

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

planning_agent = create_deep_agent(model=llm, tools=[room_tool], middleware=[TodoListMiddleware()])
result = planning_agent.invoke({"messages": [{"role": "user", "content":
    "Сначала запиши план в список задач, затем выполняй его и отмечай выполненные пункты. "
    "Задача: проверь аудитории A-101 и B-202 для групп 20, 28 и 35 человек и сохрани отчёт в report.md."}]})
show_steps(result["messages"])
print()
for item in result["todos"]:
    print(f"[{item['status']:>11}] {item['content']}")

Вызовы `write_todos` идут в начале — план, по ходу — отметки и в конце — итоговые статусы. Список задач
работает как внешняя память агента о том, что он делает: даже если ранние сообщения истории
сократятся, план останется в состоянии.

## 9. Командная строка

### 9.1. Агент пишет и запускает скрипт

`LocalShellBackend` хранит файлы на диске в папке `root_dir` и даёт агенту инструмент `execute` —
выполнение команд в командной строке. Так агент может написать скрипт, запустить его, прочитать
вывод и исправить ошибку.

Параметр `virtual_mode=True` делает `root_dir` корнем для файловых инструментов: путь `/rooms.py`
означает файл `workspace/rooms.py`, и выйти за пределы папки через `read_file` или `write_file`
нельзя. На `execute` это не распространяется: команда выполняется в настоящей командной строке
вашего компьютера, с вашими правами, и может обратиться к любым доступным вам файлам. Поэтому
агенту с командной строкой дают отдельную папку, а опасные команды — только с подтверждением
(раздел 9.2). В рабочих системах команды агента выполняют в изолированной среде — контейнере
или удалённой «песочнице».

In [ ]:
import shutil

from deepagents.backends import LocalShellBackend

WORKSPACE = ROOT / "workspace"
shutil.rmtree(WORKSPACE, ignore_errors=True)
WORKSPACE.mkdir()

shell_agent = create_deep_agent(model=llm, backend=LocalShellBackend(root_dir=str(WORKSPACE), virtual_mode=True))
result = shell_agent.invoke({"messages": [{"role": "user", "content":
    "Напиши скрипт rooms.py, который печатает таблицу аудиторий A-101 (24 места) и B-202 (40 мест) "
    "и их суммарную вместимость. Запусти его и покажи вывод."}]})
show_steps(result["messages"])
print("\nфайлы в workspace:", sorted(p.name for p in WORKSPACE.iterdir()))
print(result["messages"][-1].content)

Скрипт лежит в папке `workspace` — его можно открыть и запустить самому.

### 9.2. Подтверждение команд

Параметр `interrupt_on` подключает уже знакомый `HumanInTheLoopMiddleware` к инструментам обвязки.
Попросим подтверждать каждую команду `execute` и отклоним её.

In [ ]:
guarded_agent = create_deep_agent(model=llm, checkpointer=InMemorySaver(), interrupt_on={"execute": True},
                                  backend=LocalShellBackend(root_dir=str(WORKSPACE), virtual_mode=True))
thread = {"configurable": {"thread_id": "shell-1"}}
paused = guarded_agent.invoke({"messages": [{"role": "user", "content": "Выполни команду: ls -la"}]}, thread)
request = paused["__interrupt__"][0].value["action_requests"][0]
print("ждёт решения:", request["name"], request["args"])

resumed = guarded_agent.invoke(Command(resume={"decisions": [{"type": "reject"}]}), thread)
print(resumed["messages"][-1].content)

Команда не выполнена, и агент сообщил об этом. С решением `approve` команда выполнилась бы.

## 10. Субагенты

Инструмент `task` запускает **субагента** — отдельного агента со своим системным промптом, своими
инструментами и **своей историей сообщений**. Главный агент передаёт ему задание текстом и получает
обратно только итог. Все промежуточные шаги субагента остаются в его истории и не занимают контекст
главного агента.

Субагентов описывают словарями: имя, описание (по нему главный агент решает, кому что поручить),
системный промпт и инструменты. Кроме описанных, у обвязки есть субагент общего назначения
`general-purpose` с теми же инструментами, что у главного агента.

In [ ]:
room_checker = {
    "name": "room-checker",
    "description": "Проверяет вместимость и наличие проектора в аудиториях по справочнику.",
    "system_prompt": "Проверяй аудитории через get_room_info и отвечай кратким списком фактов.",
    "tools": [room_tool],
}
lead_agent = create_deep_agent(model=llm, subagents=[room_checker])
result = lead_agent.invoke({"messages": [{"role": "user", "content":
    "Поручи проверку аудиторий A-101 и B-202 субагенту room-checker и скажи, где поместятся 28 человек."}]})
show_steps(result["messages"])

В истории главного агента — один вызов `task` с заданием и один `ToolMessage` с итогом. Вызовы
`get_room_info` сделал субагент, и в эту историю они не попали. Так главный агент может поручить
тяжёлую работу — например, прочитать десяток файлов — и получить короткую выжимку.

## 11. Память и skills

### 11.1. Файл `AGENTS.md`

Параметр `memory` указывает файлы, которые обвязка загружает в системный промпт в начале работы.
Обычно это `AGENTS.md` — правила проекта: как называть файлы, в каком формате отвечать, что нельзя
делать. Агент может и сам дописывать туда то, что узнал, — так знания переживают перезапуск.

Пути здесь — пути бэкенда. Используем `FilesystemBackend` с папкой `workspace`.

In [ ]:
from deepagents.backends import FilesystemBackend

(WORKSPACE / "AGENTS.md").write_text(
    "# Правила учебной части\n"
    "- Отвечай одним предложением.\n"
    "- Номер аудитории всегда пиши в кавычках-ёлочках, например «A-101».\n",
    encoding="utf-8",
)
memory_agent = create_deep_agent(model=llm, tools=[room_tool], memory=["/AGENTS.md"],
                                 backend=FilesystemBackend(root_dir=str(WORKSPACE), virtual_mode=True))
result = memory_agent.invoke({"messages": [{"role": "user", "content": "Где поместятся 28 человек: в A-101 или в B-202?"}]})
print(result["messages"][-1].content)

Ответ следует правилам из файла, хотя в вопросе о них ничего нет. Правила в `AGENTS.md` стоит писать
однозначно: модель исполняет их буквально.

### 11.2. Skills

**Skill** (навык) — папка с файлом `SKILL.md` и, при необходимости, скриптами и справочными
материалами. В заголовке `SKILL.md` — имя и описание:

```markdown
---
name: room-report
description: Отчёт о подборе аудитории для группы. Использовать, когда просят подобрать аудиторию.
---

# Отчёт о подборе аудитории
1. Проверь каждую аудиторию через get_room_info.
2. Запиши /report.md: таблица «аудитория | мест | проектор | подходит».
3. Последней строкой напиши: «Проверено по справочнику».
```

Агент с параметром `skills=["/skills/"]` видит в системном промпте только список навыков — имя и
описание каждого. Полный `SKILL.md` он читает инструментом `read_file`, когда решит, что навык
подходит к задаче. Такой приём называют **прогрессивным раскрытием**: подробные инструкции занимают
контекст, только когда нужны.

Skills — тема отдельного занятия о знаниях агента. Формат описан в спецификации
[agentskills.io](https://agentskills.io/specification), руководство по skills в Deep Agents —
в [документации](https://docs.langchain.com/oss/python/deepagents/skills).

## 12. Практика

### 12.1. Ошибка инструмента — модели, а не пользователю

В разделе 2.2 вопрос про аудиторию C-303 прервал запуск агента исключением. Напишите middleware
`tool_errors_to_model`, который перехватывает `ValueError` при вызове инструмента и возвращает модели
`ToolMessage` с текстом ошибки. Тогда модель узнает, что аудитории нет в справочнике, и ответит
пользователю сама.

Подсказки:

1. Нужен хук `wrap_tool_call`: внутри вызовите `handler(request)` в блоке `try`.
2. В `except` верните `ToolMessage`: `content` — текст ошибки, `tool_call_id` — `request.tool_call["id"]`,
   `name` — `request.tool_call["name"]`, `status="error"`.
3. Создайте агента с этим middleware и спросите: «Сколько мест в C-303?».

Критерии: запуск не прерывается; в истории есть `ToolMessage` с текстом ошибки; модель отвечает, что
такой аудитории в справочнике нет, и называет доступные.

In [ ]:
# TODO: напишите middleware tool_errors_to_model на хуке wrap_tool_call.
# TODO: создайте агента с этим middleware и спросите про аудиторию C-303.
# TODO: выведите историю через show_steps.

<details><summary>Решение — после собственной попытки</summary>

Готовое решение — в файле [solutions/tool_errors.py](solutions/tool_errors.py):

```python
from solutions.tool_errors import tool_errors_to_model

checked_agent = create_agent(model=llm, tools=[room_tool], middleware=[tool_errors_to_model])
show_steps(checked_agent.invoke({"messages": [{"role": "user", "content": "Сколько мест в C-303?"}]})["messages"])
```

</details>

Перехват ошибки в middleware отделяет обработку сбоев от самих инструментов: одно правило работает
для всех инструментов агента. В LangChain есть и встроенный `ToolErrorMiddleware` — ему передают
функцию `on_error`, которая решает, что вернуть модели.

### 12.2. Для самостоятельной работы

1. Соберите Deep Agent с командной строкой (раздел 9) и планом (раздел 8). Дайте задачу: для групп
   20, 28 и 35 человек подобрать аудиторию, написать скрипт `pick_rooms.py`, который считает свободные
   места, запустить его и сохранить отчёт `report.md`. Посмотрите план и файлы в `workspace`.
2. Добавьте этому агенту подтверждение `write_file` через `interrupt_on`. Отклоните первую запись и
   посмотрите, как агент на это реагирует.
3. Напишите middleware на `before_model`, который печатает число сообщений и общий размер истории в
   символах. Запустите с ним Deep Agent и агента из раздела 2 на одном вопросе и сравните.
4. Добавьте в `AGENTS.md` правило про формат отчёта и проверьте, что агент его соблюдает.
5. Сравните историю главного агента с субагентом и без него на одной задаче: сколько сообщений и
   символов в каждой?

## 13. Итоги

### Что запомнить

- `create_agent` строит агентный цикл, который на прошлом занятии писали руками: модель → инструменты
  → модель, пока модель не ответит текстом.
- Агент — граф LangGraph с узлами `model` и `tools`. Память диалога — checkpointer и `thread_id`.
- Middleware — единый способ управлять агентом: хуки `before_*`, `after_*` и обёртки `wrap_*` вокруг
  вызовов модели и инструментов.
- Встроенные middleware закрывают частые задачи: лимиты, подтверждение человеком, суммаризацию,
  повторы.
- Под конкретную модель агента подстраивают middleware: у GigaChat нет режима «любая функция»,
  системное сообщение — только первым, объединения типов адаптер не принимает.
- Deep Agents — обвязка для длинных задач: файлы и командная строка, план, субагенты, память, skills.
  Для GigaChat к ней автоматически подключается профиль `deepagents-gigachat`.
- Агент с командной строкой исполняет настоящие команды на вашем компьютере: отдельная папка и
  подтверждение опасных действий обязательны.

### Самопроверка

<details><summary>1. Чем агент из <code>create_agent</code> отличается от цикла прошлого занятия?</summary>

Шаги те же: вызов модели, исполнение функций, возврат результатов, повтор. Цикл ведёт граф LangGraph,
он же хранит историю в состоянии и решает, когда остановиться.
</details>

<details><summary>2. Почему без checkpointer агент не помнит прошлый вопрос?</summary>

Без checkpointer каждый `invoke` начинается с пустого состояния. Checkpointer сохраняет состояние
графа для каждого `thread_id`, и следующий вызов с тем же `thread_id` продолжает историю.
</details>

<details><summary>3. Какой хук выбрать, чтобы обработать ошибку инструмента? Чтобы сократить историю перед запросом?</summary>

Ошибку — `wrap_tool_call`: он оборачивает вызов и может перехватить исключение. Историю — `before_model`:
он получает состояние перед каждым запросом к модели и может его изменить.
</details>

<details><summary>4. Почему middleware «требовать вызов функции» должен смотреть на последнее сообщение?</summary>

`wrap_model_call` срабатывает на каждом запросе к модели, в том числе на последнем, когда результаты
функций уже получены. Если требовать вызов и там, агент не сможет ответить текстом и не закончит работу.
</details>

<details><summary>5. Зачем субагенту отдельная история?</summary>

Промежуточные шаги субагента не попадают в контекст главного агента: главный получает только итог.
Так длинная работа не раздувает его историю.
</details>

### Дальше

- [Агенты LangChain](https://docs.langchain.com/oss/python/langchain/agents) и
  [middleware](https://docs.langchain.com/oss/python/langchain/middleware/overview).
- [Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) и
  [профиль GigaChat](https://github.com/ai-forever/deepagents-gigachat).
- Статьи: [«Deep Agents: open source-обвязка на LangGraph»](https://habr.com/ru/companies/sberbank/articles/1071036/),
  [«Профиль GigaChat для Deep Agents»](https://habr.com/ru/companies/sberbank/articles/1073668/).
- Остальные ссылки — в [SOURCES.md](SOURCES.md).

Папку `workspace` с файлами агентов можно удалить после занятия.